# 📈 عدم‌قطعیت Bootstrap در Climatology Engine

این نوت‌بوک روش Bootstrap را برای تخمین عدم‌قطعیت پارامترهای توزیع معرفی می‌کند.

**مواردی که یاد می‌گیرید:**
- مفهوم Bootstrap و کاربرد آن در آمار
- روش پارامتریک Bootstrap برای تخمین عدم‌قطعیت
- محاسبه بازه‌های اطمینان (Confidence Intervals)
- تفسیر نتایج Bootstrap
- رسم نمودارهای عدم‌قطعیت
- مقایسه عدم‌قطعیت مدل‌های مختلف

---

## 📐 نظریه Bootstrap

Bootstrap یک روش بازنمونه‌گیری (Resampling) است که برای تخمین توزیع نمونه‌گیری یک آماره استفاده می‌شود.

### روش پارامتریک Bootstrap

۱. برازش توزیع بر روی داده اصلی برای تخمین پارامترها $\hat{\theta}$
۲. تولید $B$ نمونه تصادفی از توزیع برازش شده
۳. برازش توزیع بر روی هر نمونه Bootstrap برای بدست آوردن $\hat{\theta}^{(b)}$
۴. محاسبه بازه اطمینان از توزیع Bootstrap

### بازه اطمینان درصدی (Percentile CI)

$$
CI_{95\%}(\theta) = [\theta_{(0.025)}, \theta_{(0.975)}]
$$

### پارامترهای قابل تنظیم

| پارامتر | مقدار پیش‌فرض | توضیح |
|---------|---------------|-------|
| `n_bootstrap` | ۱۰۰ | تعداد تکرارهای Bootstrap |
| `confidence_level` | ۰.۹۵ | سطح اطمینان (۹۵%) |
| `random_seed` | ۴۲ | دانه تصادفی برای تکرارپذیری |

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from core.engine.plugin_loader import load_plugins
from core.uncertainty.bootstrap import bootstrap_fit

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ کتابخانه‌ها بارگذاری شدند.')

In [ ]:
# بارگذاری پلاگین‌های توزیع
plugins = load_plugins()
print(f'✅ تعداد توزیع‌های بارگذاری شده: {len(plugins)}')

for code, dist in plugins.items():
    print(f"   [{code}] {dist.name} (params: {dist.params})")

distributions = {dist.name: dist for dist in plugins.values()}

In [ ]:
# بارگذاری داده نمونه
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values

# انتخاب داده tmean برای یک سال
data_year = data[:365, 1]

print(f'📊 تعداد داده‌ها: {len(data_year)}')
print(f'   میانگین: {np.mean(data_year):.2f}°C')
print(f'   انحراف معیار: {np.std(data_year):.2f}°C')
print(f'   حجم نمونه (n): {len(data_year)}')

In [ ]:
# تابع برازش برای Bootstrap
def fit_func(data):
    """تابع برازش برای استفاده در Bootstrap"""
    dist = distributions['Normal']
    return dist.fit(data)

print("✅ تابع برازش تعریف شد.")

In [ ]:
# اجرای Bootstrap
n_bootstrap = 100  # برای سرعت (در عمل ۱۰۰-۱۰۰۰)
print(f"\n🔄 در حال اجرای Bootstrap با {n_bootstrap} تکرار...")

cis = bootstrap_fit(data_year, fit_func, n_bootstrap=n_bootstrap, confidence=0.95)

print("\n📊 نتایج Bootstrap (95% CI):")
print("=" * 60)
for param, ci in cis.items():
    print(f"\n{param}:")
    print(f"   مقدار تخمینی: {ci['mean']:.4f}")
    print(f"   بازه اطمینان: [{ci['lower']:.4f}, {ci['upper']:.4f}]")
    print(f"   عرض بازه: {ci['upper'] - ci['lower']:.4f}")
print("=" * 60)

In [ ]:
# رسم نمودار بازه اطمینان Bootstrap
fig, ax = plt.subplots(figsize=(10, 6))

params = list(cis.keys())
means = [cis[p]['mean'] for p in params]
lowers = [cis[p]['lower'] for p in params]
uppers = [cis[p]['upper'] for p in params]
errors = [means[i] - lowers[i] for i in range(len(params))]
errors_upper = [uppers[i] - means[i] for i in range(len(params))]

y_pos = np.arange(len(params))

ax.errorbar(means, y_pos, xerr=[errors, errors_upper],
            fmt='o', color='blue', capsize=8, capthick=2, 
            elinewidth=2, markersize=12, markeredgecolor='black')

ax.set_yticks(y_pos)
ax.set_yticklabels(params)
ax.set_xlabel('مقدار پارامتر', fontsize=12)
ax.set_title('فاصله اطمینان ۹۵% پارامترها (Bootstrap)', fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linestyle='-', alpha=0.2)
ax.grid(True, alpha=0.3, axis='x')

for i, (param, mean_val) in enumerate(zip(params, means)):
    ax.text(mean_val + 0.1, i, f'{mean_val:.3f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# بررسی توزیع Bootstrap برای یک پارامتر خاص
def bootstrap_distribution_for_param(data, param_name, n_bootstrap=200):
    """
    محاسبه توزیع Bootstrap برای یک پارامتر خاص
    """
    n = len(data)
    values = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=n, replace=True)
        res = fit_func(sample)
        if param_name in res and not np.isnan(res[param_name]):
            values.append(res[param_name])
    return np.array(values)

# انتخاب یک پارامتر (مثلاً p1 = mean)
param_to_plot = 'p1'  # برای توزیع نرمال، p1 میانگین است
bootstrap_vals = bootstrap_distribution_for_param(data_year, param_to_plot, n_bootstrap=200)

print(f"📊 توزیع Bootstrap برای پارامتر '{param_to_plot}':")
print(f"   تعداد نمونه‌ها: {len(bootstrap_vals)}")
print(f"   میانگین: {np.mean(bootstrap_vals):.4f}")
print(f"   انحراف معیار: {np.std(bootstrap_vals):.4f}")
print(f"   CI 95%: [{np.percentile(bootstrap_vals, 2.5):.4f}, {np.percentile(bootstrap_vals, 97.5):.4f}]")

In [ ]:
# رسم هیستوگرام توزیع Bootstrap
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(bootstrap_vals, bins=30, density=True, alpha=0.6, 
        color='blue', edgecolor='black', label='Bootstrap Distribution')

# رسم بازه اطمینان
lower = np.percentile(bootstrap_vals, 2.5)
upper = np.percentile(bootstrap_vals, 97.5)
mean_val = np.mean(bootstrap_vals)

ax.axvline(mean_val, color='red', linestyle='-', linewidth=2.5, label=f'میانگین = {mean_val:.3f}')
ax.axvline(lower, color='green', linestyle='--', linewidth=2, label=f'CI 95%: [{lower:.3f}, {upper:.3f}]')
ax.axvline(upper, color='green', linestyle='--', linewidth=2)

ax.set_xlabel(f'مقدار پارامتر {param_to_plot}', fontsize=12)
ax.set_ylabel('چگالی احتمال', fontsize=12)
ax.set_title(f'توزیع Bootstrap برای پارامتر {param_to_plot} (توزیع نرمال)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# مقایسه عدم‌قطعیت مدل‌های مختلف
print("\n🔄 محاسبه عدم‌قطعیت برای مدل‌های مختلف...")

uncertainty_results = []
for name, dist in distributions.items():
    try:
        def fit_func_dist(data):
            return dist.fit(data)
        
        cis_dist = bootstrap_fit(data_year, fit_func_dist, n_bootstrap=50, confidence=0.95)
        # محاسبه میانگین عرض بازه اطمینان
        avg_width = np.mean([ci['upper'] - ci['lower'] for ci in cis_dist.values()])
        uncertainty_results.append({
            'مدل': name,
            'عدم‌قطعیت متوسط': avg_width,
            'تعداد پارامتر': len(cis_dist)
        })
        print(f"✅ {name}: عدم‌قطعیت متوسط = {avg_width:.4f}")
    except Exception as e:
        print(f"❌ {name}: خطا - {str(e)}")

uncertainty_df = pd.DataFrame(uncertainty_results).sort_values('عدم‌قطعیت متوسط')
uncertainty_df

In [ ]:
# رسم نمودار مقایسه عدم‌قطعیت مدل‌ها
if not uncertainty_df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))

    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(uncertainty_df))]
    bars = ax.bar(uncertainty_df['مدل'], uncertainty_df['عدم‌قطعیت متوسط'], 
                  color=colors, alpha=0.7, edgecolor='black', linewidth=1)

    ax.set_xlabel('مدل', fontsize=12)
    ax.set_ylabel('عدم‌قطعیت متوسط', fontsize=12)
    ax.set_title('مقایسه عدم‌قطعیت مدل‌های مختلف', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    for bar, val in zip(bars, uncertainty_df['عدم‌قطعیت متوسط']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()
else:
    print("❌ داده‌ای برای رسم وجود ندارد.")

In [ ]:
# تأثیر تعداد تکرارهای Bootstrap بر عدم‌قطعیت
print("\n📊 تأثیر تعداد تکرارهای Bootstrap:")

n_iterations = [10, 20, 50, 100, 200]
avg_widths = []

for n in n_iterations:
    try:
        cis = bootstrap_fit(data_year, fit_func, n_bootstrap=n, confidence=0.95)
        avg_width = np.mean([ci['upper'] - ci['lower'] for ci in cis.values()])
        avg_widths.append(avg_width)
        print(f"   {n} تکرار: عدم‌قطعیت متوسط = {avg_width:.4f}")
    except Exception as e:
        print(f"   {n} تکرار: خطا - {str(e)}")
        avg_widths.append(np.nan)

if len(avg_widths) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(n_iterations, avg_widths, 'bo-', linewidth=2, markersize=8, 
            markeredgecolor='black', markeredgewidth=1)
    ax.set_xlabel('تعداد تکرارهای Bootstrap', fontsize=12)
    ax.set_ylabel('عدم‌قطعیت متوسط', fontsize=12)
    ax.set_title('تأثیر تعداد تکرارهای Bootstrap بر عدم‌قطعیت', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 📋 جمع‌بندی

در این نوت‌بوک یاد گرفتید:

✅ مفهوم Bootstrap و کاربرد آن در تخمین عدم‌قطعیت
✅ روش پارامتریک Bootstrap برای برآورد بازه اطمینان
✅ محاسبه بازه اطمینان ۹۵% برای پارامترهای توزیع
✅ رسم نمودارهای عدم‌قطعیت
✅ مقایسه عدم‌قطعیت مدل‌های مختلف
✅ تأثیر تعداد تکرارهای Bootstrap بر نتایج

---

**نکات کلیدی:**

1. Bootstrap یک روش غیرپارامتریک برای تخمین عدم‌قطعیت است.
2. هرچه تعداد تکرارهای Bootstrap بیشتر باشد، نتایج دقیق‌تر است.
3. بازه اطمینان ۹۵% نشان‌دهنده محدوده‌ای است که ۹۵% از برآوردها در آن قرار می‌گیرند.
4. مدل‌های با عدم‌قطعیت کمتر، قابل‌اعتمادتر هستند.
5. برای نتایج دقیق، حداقل ۱۰۰۰ تکرار Bootstrap توصیه می‌شود.

---

**مراحل بعدی:**
- نوت‌بوک ۰۷: پردازش موازی
- نوت‌بوک ۰۸: مصورسازی داده
- نوت‌بوک ۰۹: افزودن توزیع سفارشی